# Import

In [1]:
import requests
import os

# Downloading PDF for RAG

In [2]:
pdf_link = "https://readindiabooks.com/ebook/PDF/1567798377_Indian_History_by_Krishna_Reddy@Sarkaripost.in_(2).pdf"
temp_file_location = os.getcwd() + "/tmp/"
temp_file = "sample.pdf"

In [3]:
if not os.path.exists(temp_file_location):
    os.makedirs(temp_file_location)

In [4]:
if not os.path.exists(temp_file_location + temp_file):
    print("Downloading PDF file from:", pdf_link)
    print("Saving to:", temp_file_location + temp_file)
    with open(temp_file_location + temp_file, "wb") as f:
        response = requests.get(pdf_link)
        if ((response.status_code == 200) & (response.headers['Content-Type'] == 'application/pdf')):
            print("PDF file downloaded successfully.")
            f.write(response.content)
        else:
            print("Failed to download PDF file. Status code:", response.status_code)
            print("Content-Type:", response.headers['Content-Type'])
else:
    print("PDF file already exists at:", temp_file_location + temp_file)
    print("Skipping download.")

PDF file already exists at: c:\Users\ssrim\Documents\Projects\GIT\classmate\gen_ai/tmp/sample.pdf
Skipping download.


# Reading PDF

In [5]:
from langchain_community.document_loaders import PyPDFLoader

In [6]:
loader = PyPDFLoader(temp_file_location + temp_file)
docs = loader.load()
print(len(docs))

2230


# Exploring PDF

In [8]:
docs[0]

Document(metadata={'source': 'c:\\Users\\ssrim\\Documents\\Projects\\GIT\\classmate\\gen_ai/tmp/sample.pdf', 'page': 0}, page_content='')

In [10]:
docs[100].page_content[:500]  # Print the first 500 characters of the first page

'Mohanjodaro and Harappa, Like other Indus Valley sites, had wide roads\ncutting thorugh the buildings.\nhttp://www.ancient-civilisations.com/lesser-known-facts-indus-valley-\ncivilisation/2/\nWide roads in Mohenjodaro and other Harappan sites\nDrainage System\nThe most important salient feature is perhaps the largescale use of burnt-brick\ndrains. The drainage system seems to have been quite extensive, at least at\nMohenjodaro and Lothal. At both places, there are drains in all the larger\nstreets and qu'

# Chunking

In [11]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [12]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
split_text = text_splitter.split_documents(docs)

In [15]:
split_text[100]

Document(metadata={'source': 'c:\\Users\\ssrim\\Documents\\Projects\\GIT\\classmate\\gen_ai/tmp/sample.pdf', 'page': 57}, page_content='level at Daimabad.\nAn outstanding find of this period at Daimabad is a hoard of four heavy\nsolid-cast copper objects (weighing 60 kg in all) showing a man driving a\nchariot, a buffalo on a four-legged platform attached to four solid wheels, an\nelephant on a similar platform but with its axles and wheels missing, and\nfinally, a rhino shown standing on the axles of four solid wheels.\nThe third period at Daimabad is represented by the ‘Daimabad culture’,\nwhich covered an area of about 20 ha and had, as its diagnostic trait, ill fired,\nBlack-painted Red Ware. The Malwa cultural phase constitutes the fourth\nperiod of the Daimabad sequence. There is extensive structural evidence\nbelonging to this phase. A number of structures have been identified as\nreligious, mainly on the basis of the occurrence of fire-altars in them. This is\nalso the general 

# Embedding

In [17]:
from langchain.embeddings import HuggingFaceEmbeddings

# Choose a free open-source model
model_name = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(model_name=model_name)

C:\Users\ssrim\AppData\Local\Temp\ipykernel_9944\1480759808.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\ssrim\anaconda3\envs\llms2\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ssrim\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Vector Database

In [18]:
from langchain.vectorstores import Chroma

db = Chroma.from_documents(docs, embeddings, persist_directory="tmp\db")
db.persist()

C:\Users\ssrim\AppData\Local\Temp\ipykernel_9944\3770752237.py:4: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


In [34]:
retriever = db.as_retriever(search_type="similarity", k=40)

In [35]:
from langchain.chains import RetrievalQA
from langchain_ollama.llms import OllamaLLM

llm = OllamaLLM(model="llama3.2:latest", run_local=True)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff"   # could also be "map_reduce" or "refine"
)

In [36]:
response = qa_chain.invoke("How did the harappan civilization come to an end?")
print(response)

{'query': 'How did the harappan civilization come to an end?', 'result': "I don't know how the Harappan civilization came to an end. The provided context presents various theories and debates among scholars about the decline and transformation of the Indus Civilization, including economic factors, natural calamities, and socio-cultural erosion, but it does not provide a clear consensus or a definitive answer to this question."}


In [37]:
#Page 948
response = qa_chain.invoke("When did indian classical music start")
print(response)

{'query': 'When did indian classical music start', 'result': "I don't know when Indian classical music started. The text provides information on the development and evolution of Indian classical music, but it does not provide a specific date or time period for its origin."}


In [38]:
response = qa_chain.invoke("What is the earliest Raga?")
print(response)

{'query': 'What is the earliest Raga?', 'result': 'I don\'t know what the earliest Raga is. The text mentions that there are 72 melakarta ragas, which are considered to be the basis of Contemporary Carnatic music and are also known as "janaka ragas" or "thai" (mother) ragas. However, it does not specifically state which one of these 72 ragas is the earliest.'}


In [40]:
response = qa_chain.invoke("What are the types of Carnatic music?")
print(response)

{'query': 'What are the types of Carnatic music?', 'result': "I don't know what specific types of Carnatic music there are, as the text provided does not explicitly list them. However, it mentions that Carnatic music compositions can take various forms, such as:\n\n1. Kriti (or kirtanam): a form developed between the 14th and 20th centuries by composers like Purandara Dasa.\n2. Charanas: which can be part of different structures for a kriti, including:\n\t* Chittaswara: consisting only of notes with no words.\n\t* Madhyamakāla: sung at double speed immediately after the charana.\n\nIt also mentions that Carnatic ragas differ from Hindustani ragas and have their own unique names. However, it does not provide a comprehensive list of types or categories within Carnatic music."}


In [ ]:
# Page 954
response = qa_chain.invoke("What are the forms of Hindustani music?")
print(response)

{'query': 'What are the forms of Hindustani music?', 'result': "I don't have that information. The provided text does not mention specific forms of Hindustani music. It discusses similarities and differences between Carnatic and Hindustani music, but it does not provide a comprehensive list of forms unique to Hindustani music."}


In [ ]:
# from langchain.vectorstores import Chroma

# db = Chroma.from_documents(docs, embeddings, persist_directory="db")
# db.persist()  # save to disk


# Removing Temp Folder

In [7]:
# # Removing tmp folder if it exists
# if os.path.exists(temp_file_location):
#     os.rmdir(temp_file_location)
#     print("Temporary folder removed.")